In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 27 — FAIRNESS: EM BUSCA DE UM MODELO EQUITATIVO

> "Um modelo que não é justo não é apenas ruim — é perigoso. A otimização sem equidade é a arquitetura da injustiça."
> — Uma Filósofa da Ética em IA

Eu estava no topo da montanha: um sistema robusto, preciso, monitorável. Mas do topo a visão é mais ampla, e eu enxerguei uma sombra que não notara na subida — a da parcialidade. Meu modelo foi treinado num dataset que, como todos, é uma amostra imperfeita do mundo. Ele funcionaria igualmente bem para uma mulher de 30 anos e uma de 70? Para diferentes etnias?

Acurácia média é como o PIB de um país: parece boa no geral, mas esconde desigualdades. Um modelo pode ter 98% no agregado e 85% para um subgrupo. Isso não seria só falha técnica: seria falha moral. Eu tinha a responsabilidade de, ao menos, saber medir isso.

## O Espectro do Viés e as Métricas de Equidade

O viés pode vir dos **dados** (grupos sub-representados, preconceitos históricos), do **algoritmo** ou da **interação humana**. Para medi-lo, precisamos definir "justo". Duas definições comuns:

**Paridade Demográfica** — a taxa de previsões positivas deve ser igual entre grupos. Noção forte e muitas vezes irrealista: se um grupo tem incidência real maior, um modelo justo *não deveria* ter paridade.

**Igualdade de Oportunidade** — entre os que realmente têm a doença, a taxa de detecção correta (o **Recall**) deve ser igual entre grupos. Para diagnóstico de câncer, é **a** métrica relevante: queremos que o modelo seja igualmente bom em detectar o câncer, não importa a quem o paciente pertença.

## Auditando a Equidade

Nosso dataset não tem atributos demográficos, então **simulamos** um cenário para exercitar a auditoria. Criamos um atributo sensível sintético (grupos A e B) e **injetamos deliberadamente** um viés: para os casos malignos do grupo A, atenuamos os principais sinais de malignidade, como se o equipamento daquele grupo medisse pior. Depois, usamos a biblioteca `fairlearn` para medir o recall por grupo.

#### Código 27.1: Auditoria de Equidade com Fairlearn


In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from fairlearn.metrics import MetricFrame
from oncoclassify_utils import X_train, y_train

rng = np.random.default_rng(42)
X = X_train.reset_index(drop=True).copy()
y = y_train.reset_index(drop=True)
grupo = pd.Series(rng.choice(["A", "B"], size=len(X), p=[0.5, 0.5]), name="grupo")

# ATENCAO: este vies e 100% INJETADO por nos, para fins didaticos. O dataset
# real nao tem atributo demografico. Atenuamos o sinal maligno do grupo A em 40%.
sinais = ["worst concave points", "worst radius", "worst perimeter",
          "worst area", "mean concave points", "worst concavity"]
alvo = (y == 0) & (grupo == "A")
for col in sinais:
    X.loc[alvo, col] *= 0.60

Xtr, Xte, ytr, yte, gtr, gte = train_test_split(
    X, y, grupo, test_size=0.3, random_state=42, stratify=y)

modelo = RandomForestClassifier(random_state=42).fit(Xtr, ytr)
y_pred = modelo.predict(Xte)

quadro = MetricFrame(metrics=lambda yt, yp: recall_score(yt, yp, pos_label=0),
                     y_true=yte, y_pred=y_pred, sensitive_features=gte)
print("Recall (Maligno) por grupo:")
print(quadro.by_group.round(4).to_string())
print(f"\nDiferenca de recall entre grupos: {quadro.difference():.4f}")

Recall (Maligno) por grupo:
grupo
A    0.8667
B    0.9130

Diferenca de recall entre grupos: 0.0464


O `MetricFrame` revela uma disparidade **real e relevante**: o modelo detecta 91,3% dos cânceres do grupo B, mas apenas **86,7%** do grupo A — uma diferença de **4,6 pontos** de recall. Não é uma diferença desprezível de arredondamento; é uma brecha que, num contexto clínico, significaria cânceres sistematicamente mais perdidos num grupo do que no outro. Um alerta de equidade seria acionado.

Vale a honestidade: **este viés foi injetado por nós**. O objetivo não é acusar o dataset real, mas mostrar como a ferramenta **detectaria** um viés se ele existisse — e por que essa auditoria deve ser rotina antes de qualquer implantação clínica.

### Mitigação de Viés

Detectado o problema, há técnicas de mitigação em três estágios: **pré-processamento** (reequilibrar os dados de treino), **in-processamento** (otimizar desempenho *e* equidade juntos) e **pós-processamento** (ajustar as previsões — por exemplo, *thresholds* diferentes por grupo). A escolha e os *trade-offs* (melhorar equidade às vezes reduz um pouco a acurácia global) não são decisões puramente técnicas: devem envolver *stakeholders* e representantes dos grupos afetados.

> **📌 CHECKLIST DO CAPÍTULO 27**
>
> ✅ Entende o que é viés em ML e de onde vem.
>
> ✅ Conhece **Paridade Demográfica** e **Igualdade de Oportunidade** (mesmo Recall), e por que a segunda é a relevante aqui.
>
> ✅ Executou o Código 27.1 e mediu, com `fairlearn`, uma disparidade de recall de **4,6 pontos** entre grupos.
>
> ✅ Sabe que existem técnicas de mitigação em pré, in e pós-processamento.
>
> **⚠️ CRÍTICO:** Equidade é um problema sociotécnico, não um "conserto". A escolha de qual métrica otimizar e quais *trade-offs* aceitar deve ser tomada com os afetados à mesa.

A auditoria foi reveladora: um modelo pode ser simultaneamente preciso e injusto. Embora eu não pudesse aplicar isso ao OncoClassify 2.0 por falta de dados demográficos, a lição ficou. Um sistema de IA responsável não é o que funciona bem — é o que funciona bem *para todos*. Eu recomendaria, com veemência, coletar dados demográficos e auditar a equidade antes de qualquer uso clínico real.

Faltava a última peça da confiança: não entre grupos, mas entre o homem e a máquina. Como pegar todas as minhas ferramentas de interpretabilidade — SHAP, LIME — e traduzi-las em algo que um médico pudesse usar na correria de uma clínica? Eu precisava pensar na **explicabilidade clínica**.
